[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/quickstart/quickstart_openai_agents.ipynb)

# DecimalAI + OpenAI Agents SDK Quickstart

**Instrument an OpenAI Agents application with 2 lines of code.**

This notebook shows how to add DecimalAI tracing to the OpenAI Agents SDK.
Agent runs, tool calls, handoffs, and guardrails are captured automatically.

**Prerequisites:** an OpenAI API key for Step 3 — that is the model the agent runs
on. A DecimalAI key is **optional**: with one, the runs are traced to your dashboard;
without one the notebook initializes the SDK with tracing switched off and still runs,
rather than stopping at the second cell. Neither missing key raises — each cell says
which one it would need.

## Step 1 — Install & Configure

In [ ]:
# Install dependencies (the [openai-agents] extra brings the OpenAI Agents SDK)
!pip install -q "decimalai[openai-agents]"

In [ ]:
import os

import decimalai

# A key is OPTIONAL here. This cell looks for one, and if it doesn't find a real
# one it initializes the SDK in offline mode instead of raising: `enabled=False`
# is a real kill switch: no client, no network call, and the openai_agents=True
# flag is ignored, so no adapter is installed and nothing is captured. The Agent
# below still runs — it just runs untraced.
#
# To send traces to your own dashboard, get a key at
# https://app.decimal.ai/settings (Settings -> General -> API keys) and either:
#   * add it as a Colab secret named DECIMAL_API_KEY (the key icon, left sidebar),
#   * export DECIMAL_API_KEY before launching Jupyter, or
#   * flip ASK_FOR_KEY to True below and paste it into the hidden prompt.
ASK_FOR_KEY = False

# The literal this notebook used to assign to DECIMAL_API_KEY. Treated as "no key"
# rather than passed through, so a reader who pastes the snippet from the docs and
# forgets to edit it lands in offline mode instead of on a 401.
PLACEHOLDER = "dai_sk_..."


def find_key(*names) -> str:
    """First real key among env vars, Colab secrets, and (opt-in) a prompt."""
    for name in names:
        value = (os.environ.get(name) or "").strip()
        if value and value != PLACEHOLDER:
            return value

    try:
        from google.colab import userdata  # only exists inside Colab
    except ImportError:
        pass
    else:
        for name in names:
            try:
                value = (userdata.get(name) or "").strip()
            except Exception:
                continue  # secret not set, or notebook denied access to it
            if value and value != PLACEHOLDER:
                return value

    if ASK_FOR_KEY:
        import getpass
        # getpass, never input(): a key typed into a cell is a key in the
        # browser history and in the .ipynb you later share.
        return getpass.getpass(f"{names[0]} (input is hidden): ").strip()

    return ""


# DECIMALAI_API_KEY is the alias the CLI also accepts; init() reads both.
API_KEY = find_key("DECIMAL_API_KEY", "DECIMALAI_API_KEY")
ONLINE = False

if API_KEY:
    try:
        decimalai.init(api_key=API_KEY, openai_agents=True)
        ONLINE = True
    except Exception as exc:
        # init(verify=True) raises on 401/403 or an unreachable backend. An
        # expired key is not a reason to end the notebook in a traceback.
        print(f"!  that key was rejected: {type(exc).__name__}: {exc}")
        print("!  falling back to offline mode — every cell below still runs.\n")

if not ONLINE:
    decimalai.init(enabled=False)

# The LLM key is a different question. A DecimalAI key only decides whether runs
# are recorded; without an OpenAI key there is no agent run to record at all. Same
# three places, and exported to the environment because that is where the LLM
# client looks for it.
OPENAI_KEY = find_key("OPENAI_API_KEY")
if OPENAI_KEY.endswith("..."):
    OPENAI_KEY = ""  # the placeholder this cell used to assign, not a key
if OPENAI_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY
LLM_READY = bool(OPENAI_KEY)

print("DecimalAI: ONLINE  — Agent runs will be traced to https://app.decimal.ai/traces"
      if ONLINE else
      "DecimalAI: OFFLINE — no usable key, so the SDK is initialized with tracing off.\n"
      "                     Nothing is captured; every cell below still runs.")
print("OpenAI:    ready — Step 3 will call the model." if LLM_READY else
      "OpenAI:    missing — Step 3 drives a real LLM, so it will print what it skipped\n"
      "                     instead of running. Set OPENAI_API_KEY (env var or Colab\n"
      "                     secret) and re-run this cell.")

## Step 2 — Define an Agent with Tools

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def search_docs(query: str) -> str:
    """Search the knowledge base for relevant articles."""
    return f"Found 3 results for '{query}': [Article 1, Article 2, Article 3]"


@function_tool
def check_order(order_id: str) -> str:
    """Look up an order status by order ID."""
    return f"Order {order_id}: Shipped on April 25, arriving April 29."


# Defining the agent needs no key — it is a description of one. The key is what
# running it costs, so the guard is in Step 3.
agent = Agent(
    name="support-agent",
    instructions="You are a helpful customer support assistant. Use the tools to answer questions.",
    tools=[search_docs, check_order],
)

print("OpenAI Agent ready with 2 tools")

## Step 3 — Run the Agent (Traces Are Auto-Captured)

In [ ]:
async def run_queries():
    queries = [
        "How do I reset my password?",
        "Where is my order ORD-12345?",
        "What is your return policy?",
    ]
    for q in queries:
        print(f"\nQ: {q}")
        result = await Runner.run(agent, q)
        print(f"A: {result.final_output}")

    if ONLINE:
        decimalai.flush()  # drain the background sender before you go look
        print("\n3 traces auto-captured and sent to DecimalAI!")
        print("Open your dashboard: https://app.decimal.ai/traces")
    else:
        print("\n3 agent runs finished. DecimalAI is offline, so none of them were")
        print("captured — add a key in Step 1 to see them in the dashboard.")


if LLM_READY:
    await run_queries()
else:
    print("Skipped: running the agent calls a real LLM, which needs an OpenAI key.")
    print("Set OPENAI_API_KEY (env var or Colab secret), re-run Step 1, then this cell.")

## What Gets Captured

**Needs a DecimalAI key.** For each Agent run, DecimalAI records:
- Agent name, instructions, and model
- Every LLM call with token usage
- Tool call inputs and outputs
- Agent handoffs (if using multi-agent)
- Guardrail evaluations
- The full manifest (tools + model + instructions)

## Next Steps

- 📖 [Main Quickstart](./quickstart.ipynb) — See the version-aware manifest loop (runs with no keys at all)
- 📖 [Concepts](https://docs.decimal.ai/concepts) — Understand manifests, traces, and datasets
- 📖 [Dashboard](https://app.decimal.ai) — Explore your traces